# Training Notebook for DeepPlateau
## Minimal Discs in $\mathbb{H}^3$

In this notebook, we train and save Physics-Informed Neural Networks (PINNs) to construct minimal discs in $\mathbb{H}^3$ that meet the $S^2$ sphere at infinity along a prescribed curve. While this notebook focuses on $\mathbb{H}^3$, the underlying framework can easily be modified to construct minimal discs in $\mathbb{H}^{n}$ for any $n \geq 2$.

In [ ]:
import numpy as np

from src.curves import *
from src.extensions import *
from src.full_models import HyperbolicMinimalSurfacePINN
from src.plotting import plot_error, plot_curve_and_projection, plot_H3_surfaces, montecarlo_error
from src.geometry import minimal_in_Hn_PDE_flat
from src.samplers import MixSampler
from src.training import train_PINN_Adam, refine_PINN_lbfgs

from mpl_toolkits.mplot3d import Axes3D
%matplotlib widget

## 1. Model Initialization and Hyperparameters

First, we initialize our PINN by defining the geometric and architectural hyperparameters. As detailed in the paper, our framework constructs simple b-maps depending on five main objects, all of which are specified in the dictionary below:

1. **Boundary Curve (`curve_type`)**: The curve at infinity $\gamma: S^1 \to \mathbb{R}^2$. You can choose from various implemented curves in `src/curves.py`.
2. **Interior Model (`interior_model_type`)**: The neural network architecture representing the learnable perturbation $NN$. We default to a Multi-Layer Perceptron (MLP).
3. **Boundary Defining Function (`bdf_type`)**: The function $\rho$ that ensures the map reaches the boundary at infinity. The *stereographic* choice yields the best empirical results.
4. **Extension Operator (`ext_type`)**: The initial analytic extension $ext(\gamma)$ into the disk. We use the *stereobiharmonic* extension to guarantee orthogonality at the boundary.
5. **Decay Exponent (`decay_exponent`)**: The integer $k \in \{1, 2\}$ that controls the boundary behavior of the network's correction.

*Note: You can optionally uncomment the `curve_perturbation_matrix` to introduce a small generic deformation to the curve.*

In [ ]:
# ------------------
# Model construction
# ------------------
model_kwargs = {
    # 1. knot
    'curve_type': 'gear',
    'curve_kwargs': {
        'R':2.0,
        'amp':0.5,
        'sharpness':3.0,
        'teeth':6.0,
    },
    # 'curve_perturbation_matrix': generate_curve_perturbation_matrix(
    #     N=4,
    #     scale=0.05,
    #     seed=42,),

    # 2. interior model
    'interior_model_type': 'mlp',
    'interior_model_kwargs': {
        'in_dim': 2,
        'out_dim': 3,
        'hidden_dim': 64,
        'num_hidden_layers': 4
    },

    # 3. bdf
    'bdf_type': 'stereographic',

    # 4. extension
    'ext_type': 'stereobiharmonic',
    'ext_kwargs': {'N': 15, 'num_samples': 10000},

    # 5. decay exponent
    'decay_exponent': 2
}

# we define two identical models
# we shall train only one of them and then compare the final result to the starting surface
untrained_model = HyperbolicMinimalSurfacePINN(**model_kwargs)
model = HyperbolicMinimalSurfacePINN(**model_kwargs)

## 2. Visualizing the Boundary Condition

Before initiating the training process, it is useful to visualize the prescribed boundary curve $\gamma$ to ensure it has been configured correctly. 

The cell below extracts the curve from the initialized model and plots it within the two standard conformal models of $\mathbb{H}^3$:
* **Half-Space Model**: The boundary at infinity is represented as the flat plane $z=0$ (corresponding to $\mathbb{R}^2 \cup \{\infty\}$).
* **Poincaré Ball Model**: The boundary at infinity is represented as the unit sphere $S^2$.

*Note: The 3D plots are interactive. If your notebook environment supports it, you can change the point of view by clicking and dragging the mouse pointer.*

In [ ]:
curve = model.get_curve()
plot_curve_and_projection(curve)

## 3. Evaluating the Initial PDE Residual distribution

Before starting the optimization process, we can visualize the initial PDE error across the domain (the unit disk $D^2$). In our formulation, this error corresponds to the squared pointwise norm of the tension field, $|\tau(u)|^2$.

Notice the behavior at the edge of the disk. Thanks to our geometric model design, the boundary conditions are satisfied exactly, and the initial surface already intersects the boundary at infinity orthogonally. Consequently, the PDE error is identically zero along the boundary even *before* training begins. The neural network's only job is to minimize the residual in the interior.

In [ ]:
plot_error(
    minimal_in_Hn_PDE_flat(model),
    dtype=model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title='PDE error distribution')

## 4. Training

### 4.1. Phase 1: Stochastic Optimization with Adam

With the model initialized, we begin the first phase of the training pipeline. We start with a stochastic first-order method (**Adam**) to rapidly navigate the parameter space and bring the PDE residual down to a moderate level.

To compute the interior loss, we sample $2^{14}$ collocation points from the domain $D^2$ using the `MixSampler`. During this phase:
* **Mini-batching**: The points are partitioned into batches of $2^{10}$ to compute stochastic gradients.
* **Learning Rate Schedule**: We train for 10,000 epochs using a cosine annealing schedule, smoothly decaying the learning rate from an initial $10^{-3}$ down to a terminal $10^{-5}$. This helps the optimizer settle stably into a local minimum.

In [ ]:
n_points = 2**14
sampler = MixSampler(dtype=model.kwargs['dtype'])
xy = sampler(n_points)

best_loss, _ = train_PINN_Adam(
    model,
    xy_grid=xy,              
    epochs=10000,
    batch_size=2**10,
    lr=1e-3,
    lr_min=1e-5,
    scheduler_type='cosine',  # Choose 'plateau' or 'cosine'
    # scheduler_patience=20,    # Used only if scheduler_type = 'plateau'
    # scheduler_factor=0.9,     # Used only if scheduler_type = 'plateau'
    # scheduler_threshold=1e-2, # Used only if scheduler_type = 'plateau'
    verbose=True
)

With the Adam optimization complete, we evaluate the current state of the model by re-plotting the PDE error distribution across the disk. 

Because Adam is highly effective at quickly navigating the loss landscape, we see a significant decrease in the interior residual compared to the initial plot. The surface is now much closer to being a true minimal p-immersion, setting the stage for the final high-precision refinement.

In [ ]:
plot_error(
    minimal_in_Hn_PDE_flat(model),
    dtype=model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title='PDE error distribution')

### 4.2. Phase 2: High-Precision Refinement with L-BFGS

Now we move on to the full-batch quasi-Newton refinement using the L-BFGS algorithm. 

By switching to L-BFGS, we utilize accumulated curvature information (approximating the inverse Hessian) to achieve superlinear convergence near the smooth minimizer. 

Before running this phase, we draw a **fresh batch** of interior collocation points to evaluate the full-batch loss. We push the optimizer until it hits extremely tight tolerances:
* **Gradient norm tolerance (`tolerance_grad`)**: $10^{-12}$.
* **Parameter change tolerance (`tolerance_change`)**: $10^{-14}$.

*Note: If we used single precision, the additional rounding errors from composing forward-mode and reverse-mode automatic differentiation would cause the residual to artificially stagnate around $10^{-4}$.*

In [ ]:
xy = sampler(n_points)

_ = refine_PINN_lbfgs(
    model,
    xy_grid=xy,              
    lr=1.0,
    max_iter=10000,
    log_every=10,
    tolerance_grad=1e-12,
    tolerance_change=1e-14
)

## 5. Final Assessment

With the L-BFGS refinement complete, we perform a comprehensive evaluation of our solution. 

First, we plot the PDE error distribution one last time.

In [ ]:
plot_error(
    minimal_in_Hn_PDE_flat(model),
    dtype=model.kwargs['dtype'],
    vmin=0,
    vmax=1,
    grid_size=300,
    boundary_linewidth=2.6,
    figsize=(6,5),
    colorbar_label='',
    title='PDE error distribution')

### Monte Carlo Validation

To rigorously ensure that our model actually learned the underlying PDE and didn't just overfit to the specific collocation points used during training, we perform a Monte Carlo simulation. 

Here, we draw 1,000 independent uniform samples from the disk (each of size 16,384) and compute the average PDE loss for each batch. A tight, normally distributed histogram around a very small mean confirms that the surface is genuinely minimal everywhere, not just at the points it was trained on.

In [ ]:
montecarlo_error(
    minimal_in_Hn_PDE_flat(model),
    num_samples=1000,
    size_samples=2**14,
    figsize=(5,3),
    bins=100,
    title=None,
    xlabel = 'Loss'
)

### Surface Visualization

Finally, we visualize the 3D surfaces produced by the model, before and after training, plotted in both the half-space and Poincaré ball models of $\mathbb{H}^3$. 

The untrained model is shown in green/yellow, while the fully trained minimal surface is shown in blue/red. 

*Note: These plots are interactive. You can click and drag to change the point of view. Additionally, the `max_theta` parameters in the cell below are currently set to $2\pi - \pi/2$ to create a "cutaway" view, allowing you to see the interior cross-section of the surfaces. Change these to $2\pi$ to view the complete, closed surfaces.*

In [ ]:
plot_H3_surfaces(
    model_A_trained = model,
    grid_size_A=500,
    min_r_A = 0, 
    max_r_A = 1,
    min_theta_A = 0,
    max_theta_A = 2 * np.pi - np.pi / 2, # change this to 2 * np.pi to see the whole surface
    alpha_A = 0.8, # change this to tune the transparency of the trained surface
    
    model_B_untrained = untrained_model,
    grid_size_B=500,
    min_r_B = 0,
    max_r_B = 1,
    min_theta_B = 0,
    max_theta_B = 2 * np.pi - np.pi / 2, # change this to 2 * np.pi to see the whole surface
    alpha_B = 0.8, # change this to tune the transparency of the untrained surface
)

## 6. Saving the Trained Model

Now that we are satisfied with the trained minimal surface, we save the model's parameters to disk. This preserves the optimized weights of the neural network, allowing us to reload the exact state of the model later without needing to retrain it from scratch. 

You can load the saved weights and visualise the pre-trained surfaces in the notebook `load_and_analyse`.

In [ ]:
trained_models_path = 'trained_models'
model.save(directory=trained_models_path)